In [4]:
import pandas as pd
import csv
import os

In [16]:
def strip_object_columns(df: pd.DataFrame) -> pd.DataFrame:

    for column in df.select_dtypes(include=['object']).columns:
        df[column] = df[column].astype(str).str.strip()
    
    return df

In [18]:
def clean_numeric_columns(df: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame: # adds a threshold of 0.7 (70%); so unless the row has 70% >= non-null values that are numeric, it will execute, if not, no conversion will be done on that row 
    df = df.copy()
    
    for col in df.columns:
        if col.lower() in {}:
            continue
        
        cleaned = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        numeric = pd.to_numeric(cleaned, errors="coerce") # updated "ignore" to "coerce" due to FutureWarning

        non_null = cleaned.notna().sum()
        numeric_count = numeric.notna().sum()

        if non_null > 0 and numeric_count / non_null >= threshold: 
            df[col] = numeric
    
    return df

In [19]:
def remove_empty_rows(df: pd.DataFrame) -> pd.DataFrame: # this will remove the rows where the numeric columns are NaN
    numeric_cols = df.select_dtypes(include=['number']).columns
    
    if len(numeric_cols) == 0:
        return df # this checks the number of numeric rows; if 0, then it will end
    
    return df.dropna(subset=numeric_cols, how='all')

In [20]:
def format_numbers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    def format_whole_numbers(x):
        if pd.isna(x):
            return x
        if isinstance(x, float) and x.is_integer():
            return int(x)
        return x

    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].applymap(format_whole_numbers)

    return df

# function that formats numbers with no decimal value to whole integers and leaves float values untouched

In [21]:
def fix_headers_and_rows(df: pd.DataFrame, drop_rows=2) -> pd.DataFrame:
    
    df = df.iloc[drop_rows:].reset_index(drop=True) # drop rows

    df.columns = [ # rename/replace columns
        "Age",
        "Plan.Low Cost.Weekly",
        "Plan.Low Cost.Monthly",
        "Plan.Moderate Cost.Weekly",
        "Plan.Moderate Cost.Monthly",
        "Plan.Liberal Cost.Weekly",
        "Plan.Liberal Cost.Monthly"
    ]

    df["Sex"] = df["Age"].str.extract(r"(Female|Male)")[0].ffill() # remove the rows that are currently in the "Age" column in the original dataset and applying appropriate "Sex" to that row using forward fill (ffill()) # this also creates the "Sex" column while at the same time assigning a value to it
    df["Sex"] = df["Sex"].fillna("Child") # replace NaN with "Child"

    df = df[~df["Age"].str.endswith(":")].copy() # remove rows that end with ":"

    age_clean = ( # cleans and catches potential typos from 'messy' data or data that did not transfer well; can add more stipulations to this as well
        df["Age"].astype(str)
        .str.lower()
        .str.replace("years", "", case=False)
        .str.replace("year", "", case=False, regex=False)
        .str.replace("+", "", case=False, regex=False)
        .str.strip() 
    )
    
    age_start = (
        age_clean
    .str.split("-").str[0] # take first part of the range 
    .str.split().str[0] # take first token (text separated by spaces)
    .astype(float) # number conversion
    )

    df["Group"] = pd.Series(index=df.index, dtype='object')

    df.loc[df["Sex"] == "Child", "Group"] = "Child"
    df.loc[(age_start <= 11) & (df["Sex"] != "Child"), "Group"] = "Child"
    df.loc[(age_start.between(12, 18)) & (df["Sex"] != "Child"), "Group"] = "Teen"
    df.loc[(age_start > 18) & (df["Sex"] != "Child"), "Group"] = "Adult"

    df = df[
        ["Group", "Sex", "Age", "Plan.Low Cost.Weekly", "Plan.Low Cost.Monthly", "Plan.Moderate Cost.Weekly", "Plan.Moderate Cost.Monthly", "Plan.Liberal Cost.Weekly", "Plan.Liberal Cost.Monthly"]
    ]

    return df

In [22]:
def add_suffix_cleaned(file_name, suffix='_cleaned'): # this will add the suffix 'cleaned' to the file name separated by an underscore
    base, ext = file_name.rsplit(".", 1) 
    return f"{base}{suffix}.{ext}" # splits the name of the file into 'base' and 'ext'; adds '_cleaned' to the new file name before the file extension

In [23]:
def add_suffix_test(file_name, suffix='_test'):
    base, ext = file_name.rsplit(".", 1) 
    return f"{base}{suffix}.{ext}" 

In [ ]:
def batch__fix_headers_and_rows(input_dir: str, output_dir: str, filenames: list[str]):
    for filename in filenames:
        input_file = f"{input_dir}/{filename}"
        output_file = f"{output_dir}/{filename}"

        df = pd.read_csv(input_file, index_col=False)
        df = fix_headers_and_rows(df)
        df.to_csv(output_file, index=False)
        print(f"Cleaned {filename} to {output_file}")

In [ ]:
def rename_csv_files(input_dir: str, output_dir: str, csv_rename_files: dict): # rename csv files with selective names; no blanket names

    for old_file, new_file in csv_rename_files.items():
        input_path = f"{input_dir}/{old_file}"
        output_path = f"{output_dir}/{add_suffix_test(new_file)}"

        df = pd.read_csv(input_path)
        df.to_csv(output_path, index=False)

        print(f"Renamed {old_file} to {new_file}")

In [ ]:
def batch_clean_csv( # cleans a batch of csv files and adds '_cleaned' before the .ext to show it is finalized
    input_dir: str,
    output_dir: str,
    filenames: list[str]
    ):
    
    for filename in filenames:
        input_file = f"{input_dir}/{filename}"
        output_file = f"{output_dir}/{add_suffix_cleaned(filename)}"

        df = pd.read_csv(input_file)    
        # df = batch__fix_headers_and_rows(df)
        df = strip_object_columns(df)
        df = clean_numeric_columns(df)
        df = remove_empty_rows(df)
        df = format_numbers(df)

        df.to_csv(output_file, index=False)
        print(f"Saved cleaned CSV: {output_file}")

In [ ]:
csv_rename_files = {
    "tabula-cnpp-Cost-Food-LowModerateLiberal-FoodPlans-March2025.csv": "food_cost_plans_march_2025_test.csv",
    "tabula-cnpp-costfood-3levels-august2025.csv": "food_cost_plans_august_2025_test.csv",
    "tabula-CNPP-CostFood-3levels-July2025.csv": "food_cost_plans_july_2025_test.csv",
    "tabula-cnpp-costfood-3levels-sept2025.csv": "food_cost_plans_september_2025_test.csv",
    "tabula-cnpp-costfood-3levelsTFP-june2025.csv": "food_cost_plans_june_2025_test.csv",
    "tabula-cnpp-costfood-3levelsTFP-may2025.csv": "food_cost_plans_may_2025_test.csv",
    "tabula-cnpp-costfood-tfp-3levels-april2025.csv": "food_cost_plans_april_2025_test.csv",
    "tabula-Cost_Of_Food_Low_Moderate_Liberal_Food_Plans_Febuary_2025.csv": "food_cost_plans_february_2025_test.csv",
    "tabula-Cost_Of_Food_Low_Moderate_Liberal_Food_Plans_January_2025.csv": "food_cost_plans_january_2025_test.csv"
}

input_dir = "../data/food/working/test"
output_dir = "../data/food/working/test"

rename_csv_files(input_dir, output_dir, csv_rename_files)

Renamed tabula-cnpp-Cost-Food-LowModerateLiberal-FoodPlans-March2025_test.csv to food_cost_plans_march_2025.csv
Renamed tabula-cnpp-costfood-3levels-august2025_test.csv to food_cost_plans_august_2025.csv
Renamed tabula-CNPP-CostFood-3levels-July2025_test.csv to food_cost_plans_july_2025.csv
Renamed tabula-cnpp-costfood-3levels-sept2025_test.csv to food_cost_plans_september_2025.csv
Renamed tabula-cnpp-costfood-3levelsTFP-june2025_test.csv to food_cost_plans_june_2025.csv
Renamed tabula-cnpp-costfood-3levelsTFP-may2025_test.csv to food_cost_plans_may_2025.csv
Renamed tabula-cnpp-costfood-tfp-3levels-april2025_test.csv to food_cost_plans_april_2025.csv
Renamed tabula-Cost_Of_Food_Low_Moderate_Liberal_Food_Plans_Febuary_2025_test.csv to food_cost_plans_february_2025.csv
Renamed tabula-Cost_Of_Food_Low_Moderate_Liberal_Food_Plans_January_2025_test.csv to food_cost_plans_january_2025.csv


In [10]:
fix_headers_and_rows_files = [
    "tabula-cnpp-Cost-Food-LowModerateLiberal-FoodPlans-March2025.csv",
    "tabula-cnpp-costfood-3levels-august2025.csv",
    "tabula-CNPP-CostFood-3levels-July2025.csv",
    "tabula-cnpp-costfood-3levels-sept2025.csv",
    "tabula-cnpp-costfood-3levelsTFP-june2025.csv",
    "tabula-cnpp-costfood-3levelsTFP-may2025.csv",
    "tabula-cnpp-costfood-tfp-3levels-april2025.csv",
    "tabula-Cost_Of_Food_Low_Moderate_Liberal_Food_Plans_Febuary_2025.csv",
    "tabula-Cost_Of_Food_Low_Moderate_Liberal_Food_Plans_January_2025.csv"
]
input_dir = "../data/food/working/2025"
output_dir = "../data/food/working/test"

batch__fix_headers_and_rows(input_dir, output_dir, fix_headers_and_rows_files)

Cleaned tabula-cnpp-Cost-Food-LowModerateLiberal-FoodPlans-March2025.csv to ../data/food/working/test/tabula-cnpp-Cost-Food-LowModerateLiberal-FoodPlans-March2025_test.csv
Cleaned tabula-cnpp-costfood-3levels-august2025.csv to ../data/food/working/test/tabula-cnpp-costfood-3levels-august2025_test.csv
Cleaned tabula-CNPP-CostFood-3levels-July2025.csv to ../data/food/working/test/tabula-CNPP-CostFood-3levels-July2025_test.csv
Cleaned tabula-cnpp-costfood-3levels-sept2025.csv to ../data/food/working/test/tabula-cnpp-costfood-3levels-sept2025_test.csv
Cleaned tabula-cnpp-costfood-3levelsTFP-june2025.csv to ../data/food/working/test/tabula-cnpp-costfood-3levelsTFP-june2025_test.csv
Cleaned tabula-cnpp-costfood-3levelsTFP-may2025.csv to ../data/food/working/test/tabula-cnpp-costfood-3levelsTFP-may2025_test.csv
Cleaned tabula-cnpp-costfood-tfp-3levels-april2025.csv to ../data/food/working/test/tabula-cnpp-costfood-tfp-3levels-april2025_test.csv
Cleaned tabula-Cost_Of_Food_Low_Moderate_Liberal

In [27]:
csv_files = [
    "food_cost_plans_march_2025.csv",
    "food_cost_plans_august_2025.csv",
    "food_cost_plans_july_2025.csv",
    "food_cost_plans_september_2025.csv",
    "food_cost_plans_june_2025.csv",
    "food_cost_plans_may_2025.csv",
    "food_cost_plans_april_2025.csv",
    "food_cost_plans_february_2025.csv",
]

input_dir = "../data/food/working/test"
output_dir = "../data/food/cleaned"

batch_clean_csv(input_dir, output_dir, csv_files)

Saved cleaned CSV: ../data/food/cleaned/food_cost_plans_march_2025_cleaned.csv
Saved cleaned CSV: ../data/food/cleaned/food_cost_plans_august_2025_cleaned.csv
Saved cleaned CSV: ../data/food/cleaned/food_cost_plans_july_2025_cleaned.csv
Saved cleaned CSV: ../data/food/cleaned/food_cost_plans_september_2025_cleaned.csv
Saved cleaned CSV: ../data/food/cleaned/food_cost_plans_june_2025_cleaned.csv
Saved cleaned CSV: ../data/food/cleaned/food_cost_plans_may_2025_cleaned.csv
Saved cleaned CSV: ../data/food/cleaned/food_cost_plans_april_2025_cleaned.csv
Saved cleaned CSV: ../data/food/cleaned/food_cost_plans_february_2025_cleaned.csv


C:\Users\Kyle\AppData\Local\Temp\ipykernel_9072\3887946779.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[numeric_cols] = df[numeric_cols].applymap(format_whole_numbers)
C:\Users\Kyle\AppData\Local\Temp\ipykernel_9072\3887946779.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[numeric_cols] = df[numeric_cols].applymap(format_whole_numbers)
C:\Users\Kyle\AppData\Local\Temp\ipykernel_9072\3887946779.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[numeric_cols] = df[numeric_cols].applymap(format_whole_numbers)
C:\Users\Kyle\AppData\Local\Temp\ipykernel_9072\3887946779.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[numeric_cols] = df[numeric_cols].applymap(format_whole_numbers)
C:\Users\Kyle\AppData\Local\Temp\ipykernel_9072\3887946779.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFra

In [ ]:
df = pd.read_csv("../data/food/working/2025/tabula-Cost_Of_Food_Low_Moderate_Liberal_Food_Plans_Febuary_2025.csv")

result = fix_headers_and_rows(df)

result.to_csv("../data/food/working/2025/tabula-Cost_Of_Food_Low_Moderate_Liberal_Food_Plans_Febuary_2025_tested.csv", index=False)